# Clinical Change-from-Baseline Table with `by="Analysis"`

A worked example of the classic clinical descriptive-statistics layout:

> For each **Visit**, by **Treatment Arm**, show `n`, `Mean (SD)`,
> `Median (Q1, Q3)`, and `Min, Max` — for both the observed **Value** and
> **Change from Baseline** — as separate rows under each visit, with Arm
> pivoted into columns.

The key insight: two `.analyze_by(label=...)` calls on the same tree naturally
separate the two metrics at the tree level. `by="Analysis"` (new in #70) then
pivots across those labels, turning them into side-by-side columns alongside the
real `Arm` split — one tree, one `format_statistics()` call, one `simple_table()`.


In [1]:
import numpy as np
import pandas as pd
from great_tables import GT

from pyMyriad import AnalysisTree, change_from_baseline, format_statistics, simple_table

## 1. Simulate CDISC-like lab data

One row per subject per visit. `AVISIT` and `ARM` are ordered categoricals so
columns/rows come out in the intended clinical order rather than alphabetically
(see #61).


In [2]:
rng = np.random.default_rng(42)

ARMS   = ["Placebo", "Active 10 mg"]
VISITS = ["Baseline", "Week 4", "Week 8", "Week 12"]
N_SUBJECTS = 40

subjects = pd.DataFrame({
    "USUBJID": [f"S{i:03d}" for i in range(N_SUBJECTS)],
    "ARM": rng.choice(ARMS, size=N_SUBJECTS),
})

rows = []
for subj in subjects.itertuples():
    baseline = rng.normal(45, 8)
    drift = -6 if subj.ARM == "Active 10 mg" else -1
    for visit_idx, visit in enumerate(VISITS):
        aval = baseline + drift * visit_idx / 3 + rng.normal(0, 3)
        rows.append({"USUBJID": subj.USUBJID, "ARM": subj.ARM,
                     "AVISIT": visit, "AVAL": aval})

df = pd.DataFrame(rows)
df["AVISIT"] = pd.Categorical(df["AVISIT"], categories=VISITS, ordered=True)
df["ARM"]   = pd.Categorical(df["ARM"],   categories=ARMS,   ordered=True)

df = change_from_baseline(
    df,
    id_col="USUBJID",
    value_col="AVAL",
    baseline_level="Baseline",
    level_col="AVISIT",
)
df.head()

,USUBJID,ARM,AVISIT,AVAL,CHG
0,S000,Placebo,Baseline,41.478312,0.000000
1,S000,Placebo,Week 4,46.855392,5.377079
2,S000,Placebo,Week 8,42.390846,0.912534
3,S000,Placebo,Week 12,41.236118,-0.242195
4,S001,Active 10 mg,Baseline,43.779859,0.000000


## 2. Build the tree: two `analyze_by()` calls, one per metric

Each call computes the same raw statistics on a *different column* (`AVAL` vs `CHG`).
The `label=` argument tags which metric each result belongs to.  `CHG` was added
by `change_from_baseline()` in the previous cell — no long-format reshape needed.


In [3]:
tree = (
    AnalysisTree()
    .split_by("df.AVISIT", label="Visit")
    .split_by("df.ARM",    label="Arm")
    .analyze_by(
        n      = lambda df: len(df),
        mean   = lambda df: round(np.mean(df.AVAL), 1),
        sd     = lambda df: round(np.std(df.AVAL, ddof=1), 1),
        median = lambda df: round(np.median(df.AVAL), 1),
        q1     = lambda df: round(np.percentile(df.AVAL, 25), 1),
        q3     = lambda df: round(np.percentile(df.AVAL, 75), 1),
        min    = lambda df: round(np.min(df.AVAL), 1),
        max    = lambda df: round(np.max(df.AVAL), 1),
        label  = "Value",
    )
    .analyze_by(
        n      = lambda df: len(df),
        mean   = lambda df: round(np.mean(df.CHG), 1),
        sd     = lambda df: round(np.std(df.CHG, ddof=1), 1),
        median = lambda df: round(np.median(df.CHG), 1),
        q1     = lambda df: round(np.percentile(df.CHG, 25), 1),
        q3     = lambda df: round(np.percentile(df.CHG, 75), 1),
        min    = lambda df: round(np.min(df.CHG), 1),
        max    = lambda df: round(np.max(df.CHG), 1),
        label  = "Change",
    )
)
result = tree.run(df)

## 3. Combine raw statistics into display strings with `format_statistics()`

`format_statistics()` combines several raw named statistics into new formatted
string statistics in one call. `remove_original=True` drops the raw values
afterwards, leaving only the four display rows we want: `n`, `Mean (SD)`,
`Median (Q1, Q3)`, `Min, Max`.


In [4]:
formatted = format_statistics(
    result,
    n          = "{n}",
    mean_sd    = "{mean:.1f} ({sd:.2f})",
    mqq        = "{median:.2f} [{q1} - {q3}]",
    min_max    = "{min} - {max}",
    remove_original = True,
)

## 4. One `simple_table()` call produces the full layout

`by=["Arm", "Analysis"]` pivots across **both** the `Arm` split and the
analysis `label` dimension simultaneously. Columns are named `"{Arm} > {Label}"`,
giving the four-column layout (Placebo/Active × Value/Change) in a single call
with no extra reshaping.


In [5]:
table = simple_table(formatted, by=["Arm", "Analysis"], pivot_statistics=False)
table = (
    table
    .drop(columns=["_Level_0"])
    .rename(columns={"_Level_1": "Visit"})
    .rename(columns={c: c.replace(" > ", "||") for c in table.columns if " > " in c})
)
table = table[
    ["Visit", "Statistic"]
    + [f"{arm}||{measure}" for arm in ARMS for measure in ("Value", "Change")]
]
table

pivot_lvl,Visit,Statistic,Placebo||Value,Placebo||Change,Active 10 mg||Value,Active 10 mg||Change
0,Baseline,n,17,17,23,23
1,,mean_sd,45.1 (7.00),0.0 (0.00),46.2 (8.80),0.0 (0.00)
2,,mqq,41.50 [40.6 - 51.1],0.00 [0.0 - 0.0],46.70 [39.5 - 51.0],0.00 [0.0 - 0.0]
3,,min_max,30.5 - 56.1,0.0 - 0.0,29.5 - 60.9,0.0 - 0.0
4,Week 4,n,17,17,23,23
5,,mean_sd,43.8 (6.20),-1.4 (3.90),44.3 (8.20),-1.9 (3.10)
6,,mqq,43.50 [38.6 - 48.6],-2.20 [-3.4 - 1.2],44.70 [39.1 - 48.0],-1.90 [-3.5 - -0.5]
7,,min_max,31.7 - 53.9,-8.3 - 5.7,28.4 - 60.1,-7.0 - 6.6
8,Week 8,n,17,17,23,23
9,,mean_sd,44.5 (6.80),-0.7 (3.70),42.9 (7.70),-3.2 (3.20)


## 5. Great Tables version with Arm spanners

In [6]:
display_table = table.copy()
display_table.loc[display_table["Visit"].duplicated(), "Visit"] = ""

arm_columns = {
    arm: [c for c in display_table.columns if c.startswith(f"{arm}||")]
    for arm in ARMS
}

gt = GT(display_table).tab_header(
    title    = "ALT (U/L) by Visit and Treatment Arm",
    subtitle = "n; Mean (SD); Median [Q1 - Q3]; Min - Max — Value and Change from Baseline",
)
for arm, cols in arm_columns.items():
    gt = gt.tab_spanner(label=arm, columns=cols)
gt = gt.cols_label(**{c: c.split("||", 1)[1] for cols in arm_columns.values() for c in cols})
gt = gt.cols_align(align="center", columns=[c for cols in arm_columns.values() for c in cols])
gt

GT(_tbl_data=pivot_lvl     Visit Statistic       Placebo||Value      Placebo||Change  \
0          Baseline         n                   17                   17   
1                     mean_sd          45.1 (7.00)           0.0 (0.00)   
2                         mqq  41.50 [40.6 - 51.1]     0.00 [0.0 - 0.0]   
3                     min_max          30.5 - 56.1            0.0 - 0.0   
4            Week 4         n                   17                   17   
5                     mean_sd          43.8 (6.20)          -1.4 (3.90)   
6                         mqq  43.50 [38.6 - 48.6]   -2.20 [-3.4 - 1.2]   
7                     min_max          31.7 - 53.9           -8.3 - 5.7   
8            Week 8         n                   17                   17   
9                     mean_sd          44.5 (6.80)          -0.7 (3.70)   
10                        mqq  45.20 [40.5 - 50.2]    0.90 [-3.5 - 2.1]   
11                    min_max          32.6 - 58.1           -7.0 - 4.7   
12          Week 12         n                   17                   17   
13                    mean_sd          43.1 (7.20)          -2.0 (3.00)   
14                        mqq  42.60 [36.7 - 49.8]  -2.30 [-3.9 - -0.2]   
15                    min_max          32.7 - 55.7           -8.2 - 2.1   

pivot_lvl  Active 10 mg||Value Active 10 mg||Change  
0                           23                   23  
1                  46.2 (8.80)           0.0 (0.00)  
2          46.70 [39.5 - 51.0]     0.00 [0.0 - 0.0]  
3                  29.5 - 60.9            0.0 - 0.0  
4                           23                   23  
5                  44.3 (8.20)          -1.9 (3.10)  
6          44.70 [39.1 - 48.0]  -1.90 [-3.5 - -0.5]  
7                  28.4 - 60.1           -7.0 - 6.6  
8                           23                   23  
9                  42.9 (7.70)          -3.2 (3.20)  
10         43.50 [38.4 - 47.4]  -4.10 [-5.7 - -0.3]  
11                 30.6 - 59.2           -9.1 - 1.2  
12                          23                   23  
13                 40.2 (7.90)          -6.0 (4.20)  
14         39.30 [35.8 - 43.9]  -5.50 [-8.6 - -3.1]  
15                 24.7 - 58.0          -16.4 - 1.8  , _body=<great_tables._gt_data.Body object at 0x11217b0e0>, _boxhead=Boxhead([ColInfo(var='Visit', type=<ColInfoTypeEnum.default: 1>, column_label='Visit', column_align='left', column_width=None), ColInfo(var='Statistic', type=<ColInfoTypeEnum.default: 1>, column_label='Statistic', column_align='center', column_width=None), ColInfo(var='Placebo||Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='center', column_width=None), ColInfo(var='Placebo||Change', type=<ColInfoTypeEnum.default: 1>, column_label='Change', column_align='center', column_width=None), ColInfo(var='Active 10 mg||Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='center', column_width=None), ColInfo(var='Active 10 mg||Change', type=<ColInfoTypeEnum.default: 1>, column_label='Change', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10a9910d0>, _spanners=Spanners([SpannerInfo(spanner_id='Placebo', spanner_level=0, spanner_label='Placebo', spanner_units=None, spanner_pattern=None, vars=['Placebo||Value', 'Placebo||Change'], built=None), SpannerInfo(spanner_id='Active 10 mg', spanner_level=0, spanner_label='Active 10 mg', spanner_units=None, spanner_pattern=None, vars=['Active 10 mg||Value', 'Active 10 mg||Change'], built=None)]), _heading=Heading(title='ALT (U/L) by Visit and Treatment Arm', subtitle='n; Mean (SD); Median [Q1 - Q3]; Min - Max — Value and Change from Baseline', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11289e7e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x11289ea80>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x11289ef90>, _formats=[], _substitutions=[], _options

## Notes

- **Before `by="Analysis"`**, building this table required either: pre-formatting
  every (metric × statistic) combination as its own named string inside
  `analyze_by()` (as in `examples/lab_change_from_baseline_table_v2.py`), then
  hand-rolling a `pd.concat()` reshape; or reshaping the raw data into long form
  with a literal `MEASURE` column. Both added significant boilerplate outside the
  tree itself.
- The `"Analysis"` sentinel works wherever `by=` is accepted:
  `simple_table()`, `cascade_table()`, and `gt_table()`. It can be combined with
  any number of real split labels, e.g. `by=["Arm", "Analysis"]` or used standalone
  as `by="Analysis"`.
- `format_statistics()` handles the stat-combining/selecting step independently of
  table layout — the two concerns stay cleanly separated.
